这是**马尔可夫预测模型 (Markov Prediction Model)** 的详细解析。

它与前面的回归、时间序列模型完全不同。前面的模型预测的是**具体的数值**（比如明年GDP是100万），而马尔可夫模型预测的是**状态的概率**（比如明年市场占有率是多少，或者机器是“正常”还是“故障”）。

---

### 一、 算法原理
**核心思想**：**“将来只与现在有关，与过去无关。”（无后效性）**

1.  **状态 (State)**：事物所处的局面（如：晴天/雨天，盈利/亏损，市场份额A/B/C）。
2.  **转移概率 (Transition Probability)**：从一个状态跳到另一个状态的可能性。
3.  **马尔可夫链**：一串状态的变化序列。
    *   如果今天下雨，明天有 70% 概率下雨，30% 概率转晴。这就是一个简单的马尔可夫模型。
    *   我们只需要算出**转移概率矩阵**，就能算出 $n$ 天后的状态分布。

---

### 二、 推导与步骤

假设系统有 $n$ 个状态。

#### 1. 划分状态
如果数据是连续数值（如销售额），必须先**离散化**（分箱）。
*   例如：<100万(状态1)，100-200万(状态2)，>200万(状态3)。

#### 2. 统计转移次数
统计历史数据中，由状态 $i$ 变成 状态 $j$ 的次数 $N_{ij}$。

#### 3. 计算转移概率矩阵 $P$
$$ p_{ij} = \frac{N_{ij}}{\sum_{k=1}^{n} N_{ik}} $$
即：（从 $i$ 变到 $j$ 的次数）除以（$i$ 出现的总次数）。
得到的矩阵 $P$ 中，每一行的和必须为 1。

#### 4. 预测未来
假设当前的状态分布向量为 $S_0$（比如 [0.3, 0.7]），$k$ 步之后的预测状态 $S_k$ 为：
$$ S_k = S_0 \times P^k $$

#### 5. 终极状态 (稳态)
如果时间足够长，系统通常会趋于一个稳定状态（不管初始状态是什么，最后的比例是固定的）。这是马尔可夫模型最迷人的地方，常用于预测长期的**市场占有率**。

---

### 三、 适用场景
1.  **市场占有率预测**：顾客会在不同品牌间“跳来跳去”，预测未来各品牌的份额。
2.  **设备状态/故障预测**：机器从“健康”到“小毛病”再到“故障”的演变。
3.  **等级/结构变化**：人才结构预测（初级、中级、高级员工的流动）、土地利用类型变化（耕地变林地、林地变建筑用地）。
4.  **作为辅助修正**：常与**灰色预测**结合（灰色马尔可夫模型），用马尔可夫来预测灰色模型的残差状态，提高精度。

---

### 四、 Python 代码框架

这份代码设计了一个通用的马尔可夫预测器。你可以输入一串历史状态序列，它会自动计算转移矩阵并进行预测。

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 解决中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

class MarkovPredictor:
    def __init__(self):
        self.states = None      # 状态列表
        self.state_dict = {}    # 状态到索引的映射
        self.P = None           # 转移概率矩阵

    def fit(self, state_sequence):
        """
        训练模型：根据历史状态序列计算转移矩阵
        :param state_sequence: list or array, 历史状态序列 (如 ['A', 'A', 'B', 'C', 'A'])
        """
        # 1. 识别所有唯一状态并排序
        self.states = sorted(list(set(state_sequence)))
        n_states = len(self.states)
        self.state_dict = {state: i for i, state in enumerate(self.states)}

        print(f"识别到 {n_states} 个状态: {self.states}")

        # 2. 初始化计数矩阵
        count_mat = np.zeros((n_states, n_states))

        # 3. 遍历序列统计转移次数 (从 i 到 j)
        for k in range(len(state_sequence) - 1):
            current_s = state_sequence[k]
            next_s = state_sequence[k+1]

            i = self.state_dict[current_s]
            j = self.state_dict[next_s]
            count_mat[i, j] += 1

        # 4. 归一化得到概率矩阵 P
        # axis=1 按行求和, keepdims=True 保持二维以便广播除法
        row_sums = count_mat.sum(axis=1, keepdims=True)

        # 处理某状态从未转移出去的情况 (避免除以0)
        # 如果某行和为0，说明该状态是终点，通常设为保持自身 (概率1)
        row_sums[row_sums == 0] = 1

        self.P = count_mat / row_sums

        print("-" * 30)
        print("转移概率矩阵构建完成。")

    def predict(self, current_status_dist, steps=1):
        """
        预测未来状态
        :param current_status_dist: 当前状态分布向量 (如 [0.3, 0.4, 0.3]) 或 单个状态名称
        :param steps: 预测未来几步
        :return: 预测后的状态分布向量
        """
        if self.P is None:
            raise ValueError("请先调用 fit() 训练模型")

        # 处理输入：如果是单个状态名称（如 'A'），转为 one-hot 向量
        n = len(self.states)
        S0 = np.zeros(n)

        if isinstance(current_status_dist, str):
            idx = self.state_dict[current_status_dist]
            S0[idx] = 1.0
        else:
            S0 = np.array(current_status_dist)
            if abs(S0.sum() - 1) > 0.01:
                print("警告: 初始状态概率和不为1，已自动归一化")
                S0 = S0 / S0.sum()

        # 矩阵乘法预测: S_k = S_0 * P^k
        # 也就是连续乘 k 次 P
        Sk = S0
        for _ in range(steps):
            Sk = np.dot(Sk, self.P)

        return Sk

    def plot_matrix(self):
        """绘制转移矩阵热力图"""
        if self.P is None: return

        plt.figure(figsize=(8, 6))
        sns.heatmap(self.P, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=self.states, yticklabels=self.states)
        plt.title('马尔可夫状态转移矩阵')
        plt.xlabel('下一时刻状态')
        plt.ylabel('当前时刻状态')
        plt.show()

# ================= 使用示例 =================

if __name__ == "__main__":
    # 场景：某城市每年最流行的交通方式变化
    # 数据：过去15年的主要交通方式记录
    # A=公交, B=地铁, C=私家车
    history_data = [
        'A', 'A', 'B', 'B', 'B', 'C', 'C', 'B',
        'C', 'C', 'C', 'B', 'A', 'B', 'C'
    ]

    # 1. 初始化并训练
    mk = MarkovPredictor()
    mk.fit(history_data)

    # 2. 展示转移矩阵
    mk.plot_matrix()

    # 3. 预测：假设今年是 'C' (私家车)，预测 3 年后的交通格局
    future_steps = 3
    current_state = 'C'

    pred_dist = mk.predict(current_state, steps=future_steps)

    print("-" * 30)
    print(f"当前状态: {current_state}")
    print(f"{future_steps} 年后的状态分布预测:")
    for state, prob in zip(mk.states, pred_dist):
        print(f"状态 {state}: {prob*100:.2f}%")

    # 4. 稳态分析 (Long-term)
    # 预测很多步之后，分布会趋于稳定
    steady_dist = mk.predict(current_state, steps=100)
    print("-" * 30)
    print("长期稳态分布 (最终市场份额):")
    print(dict(zip(mk.states, np.round(steady_dist, 4))))
```

### 代码使用的“修修补补”指南：

1.  **如何处理连续数据？**
    *   如果你的数据是具体的销售额（如 100, 120, 90...）。
    *   **必须先“分级”**。
    *   在传入 `fit` 之前，写个循环处理一下数据：
        ```python
        raw_data = [100, 120, 95, 200...]
        state_seq = []
        for x in raw_data:
            if x < 100: state_seq.append('低')
            elif x < 180: state_seq.append('中')
            else: state_seq.append('高')
        # 然后把 state_seq 传给 fit
        ```

2.  **市场占有率问题**：
    *   题目如果给的是表格，比如：
        *   2020年：A品牌30%，B品牌70%
        *   2021年：A品牌35%，B品牌65%
        *   ...
    *   这种情况**不需要统计次数**，题目通常会直接给你“转移矩阵”（或者告诉你：A用户每年有10%转到B，B有20%转到A）。
    *   如果是这种情况，你直接**手动构建矩阵**，然后赋值给 `mk.P`，直接调用 `predict` 即可。
    *   ```python
        mk = MarkovPredictor()
        mk.states = ['A', 'B']
        mk.state_dict = {'A':0, 'B':1}
        # 手动定义 P
        mk.P = np.array([[0.9, 0.1],
                         [0.2, 0.8]])
        ```

3.  **稳态计算**：
    *   我在代码里用了一个“笨办法”：`predict(steps=100)`。
    *   原理是：只要乘的次数够多，结果就不变了。这是求稳态最简单、最不易出错的方法。
    *   *数学方法*：解方程 $\pi P = \pi$（也就是求特征值为1的特征向量），比较麻烦，写代码容易错，建议直接暴力迭代。

4.  **预测残差（高级用法）**：
    *   如果你的灰色预测模型 GM(1,1) 精度不够，可以把残差划分为“正大、正小、负小、负大”几个状态，用马尔可夫预测下一个残差的状态区间，然后对 GM(1,1) 的结果进行修正。这叫**“灰色马尔可夫模型”**，是拿奖利器。